# Attempt to convert XSD files into dataframes
Looking for a way to use xml schema definition files for dynamic code generation

ENTSOE XSD files to be found here: https://gitlab.entsoe.eu/transparency/xsd

In [0]:
# xsd file referring to another xsd file, which refers to another, which again refers to 3. file
xsd_file = '/mnt/bd/openuniverselake/aisraw/entsoe_tp/xsd/iec62325-451-6-generationload_v3_0.xsd' # CONFIG!

## Quick look on xsd content

In [0]:
df = spark.read.format("xml").option("rowTag", "xs:simpleType").load(xsd_file)
display(df)

In [0]:
df = spark.read.format("xml").option("rowTag", "xs:complexType").load(xsd_file)
display(df)

## Convert simple part

In [0]:
df_simple = spark.read.format("xml").option("rowTag", "xs:simpleType").load(xsd_file)
rename_map = {c: c.replace("xs:", "").replace("_sawsdl:", "").replace("_name", "name") for c in df_simple.columns}
df_simple = df_simple.withColumnsRenamed(rename_map)
#df_simple = df_simple.withColumn("file_name", input_file_name())
df_simple = df_simple.withColumn("base", df_simple["restriction._base"])
df_simple = df_simple.withColumn("min_inclusive", df_simple["restriction.xs:minInclusive._value"])
df_simple = df_simple.withColumn("max_inclusive", df_simple["restriction.xs:maxInclusive._value"])
df_simple = df_simple.withColumn("pattern", df_simple["restriction.xs:pattern._value"])
df_simple = df_simple.drop(*["restriction"])
display(df_simple)

## Convert complex part

In [0]:
df_complex = spark.read.format("xml").option("rowTag", "xs:complexType").load(xsd_file)
rename_map = {c: c.replace("xs:", "").replace("_sawsdl:", "").replace("_name", "name") for c in df_complex.columns}
df_complex = df_complex.withColumnsRenamed(rename_map)
#df_complex = df_complex.withColumn("file_name", input_file_name())
df_complex = df_complex.withColumn("base", df_complex["simpleContent.xs:extension._base"])
df_complex = df_complex.withColumn("use", df_complex["simpleContent.xs:extension.xs:attribute._use"])

df_complex = df_complex.withColumn("sequence_element", explode_outer("sequence.xs:element")) # sequence and simpleContent are mutually exclusive
df_complex = df_complex.withColumn("element_name", df_complex["sequence_element._name"])
df_complex = df_complex.withColumn("element_min_occurs", df_complex["sequence_element._minOccurs"])
df_complex = df_complex.withColumn("element_max_occurs", df_complex["sequence_element._maxOccurs"])
df_complex = df_complex.withColumn("element_reference", df_complex["sequence_element._sawsdl:modelReference"])
df_complex = df_complex.withColumn("element_type", df_complex["sequence_element._type"])
df_complex = df_complex.drop(*["sequence", "simpleContent", "sequence_element"])
display(df_complex)

## Create temp views

In [0]:
df_complex.createOrReplaceTempView("complex")
df_simple.createOrReplaceTempView("simple")


## Query content

In [0]:
%sql
SELECT * FROM complex order by name;

In [0]:
%sql
SELECT * FROM simple ORDER BY 1;